# KD-BiLSTM chống phishing thế hệ LLM — tái hiện ý tưởng

Bài báo: *Eskandarian et al., "A lightweight defense mechanism against next-generation of phishing emails using distilled attention-augmented BiLSTM", JISA 101 (2026) 104552.*

Notebook tự chứa: các cell đầu ghi package `phishkd/` ra đĩa, sau đó chạy lần lượt:

1. Dữ liệu (Sec. 3): sinh mẫu "LLM", phân tích độ tương đồng (Fig. 2), chấm điểm mức phishing (3.2.1), khử trùng lặp (3.2.2)
2. Các baseline LSTM / BiLSTM / +SH / +MH (Tables 4–5)
3. Teacher MobileBERT và student KD-BiLSTM (Table 6)

**Trên Colab**: vào *Runtime → Change runtime type → GPU (T4/L4)* để fine-tune MobileBERT nhanh hơn.

In [ ]:
%pip install -q gensim
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Package `phishkd` (ghi ra đĩa)

In [ ]:
import os
os.makedirs('phishkd', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile phishkd/__init__.py
"""Re-implementation of 'A lightweight defense mechanism against next-generation of
phishing emails using distilled attention-augmented BiLSTM' (Eskandarian et al., JISA 2026)."""

In [ ]:
%%writefile phishkd/text.py
"""Section 4.1 - Tokenization for the non-distilled baselines (LSTM / BiLSTM / attention).

lowercase -> remove digits -> remove punctuation -> remove stopwords -> token sequence
-> Word2Vec (CBOW, 100-d) dense vectors.
"""
import re

import numpy as np
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_HTML = re.compile(r"<[^>]+>")
_DIGITS = re.compile(r"\d+")
_PUNCT = re.compile(r"[^\w\s]|_")
_WS = re.compile(r"\s+")

PAD, UNK = "<pad>", "<unk>"


def clean_html(text: str) -> str:
    """Section 4.7.2 normalisation: strip HTML tags, special chars, extra whitespace."""
    return _WS.sub(" ", _HTML.sub(" ", str(text))).strip()


def preprocess(text: str) -> list[str]:
    text = clean_html(text).lower()
    text = _DIGITS.sub(" ", text)
    text = _PUNCT.sub(" ", text)
    return [t for t in text.split() if t not in ENGLISH_STOP_WORDS and len(t) > 1]


class Word2VecVocab:
    """Trains Word2Vec on the training corpus and maps emails to padded index sequences.

    The embedding matrix is used as a (frozen) input layer, so the model input is the
    sequence matrix X in R^{T x F} with F = 100 as in the paper.
    """

    def __init__(self, dim: int = 100, max_len: int = 200, sg: int = 0, min_count: int = 2,
                 epochs: int = 10, seed: int = 42):
        self.dim, self.max_len, self.sg = dim, max_len, sg
        self.min_count, self.epochs, self.seed = min_count, epochs, seed

    def fit(self, texts):
        sents = [preprocess(t) for t in texts]
        w2v = Word2Vec(sents, vector_size=self.dim, window=5, min_count=self.min_count,
                       sg=self.sg, epochs=self.epochs, seed=self.seed, workers=4)
        self.itos = [PAD, UNK] + list(w2v.wv.index_to_key)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        emb = np.zeros((len(self.itos), self.dim), dtype=np.float32)
        emb[1] = np.random.default_rng(self.seed).normal(0, 0.1, self.dim)
        emb[2:] = w2v.wv.vectors
        self.embedding_matrix = emb
        return self

    def encode(self, texts) -> np.ndarray:
        out = np.zeros((len(texts), self.max_len), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [self.stoi.get(w, 1) for w in preprocess(t)][: self.max_len]
            out[i, : len(ids)] = ids
        return out

In [ ]:
%%writefile phishkd/augment.py
"""Section 3.2 - LLM-driven evasion behaviours used to build the "Gen" part of the corpus.

The paper prompts closed-source LLMs (OpenAI, DeepSeek) to rewrite phishing emails with
three behaviours:
  * paraphrasing     - rewrite content to avoid keyword-based signatures
  * masking          - replace explicit phishing indicators with softer business language
  * personalization  - insert recipient-specific details so it looks legitimate

`LLMAugmenter` holds the prompts and calls any `llm_fn(prompt) -> str` you plug in
(e.g. a local Ollama server, see `local_llm_fn`). `RuleAugmenter` is an offline stand-in
that imitates the three behaviours with simple rules so the whole pipeline runs without
an LLM.
"""
import json
import random
import re
import urllib.request

PROMPTS = {
    "paraphrase": (
        "Rewrite the following email so that it keeps exactly the same intent and call to "
        "action, but uses completely different wording and sentence structure. Avoid "
        "typical spam keywords. Return only the email body.\n\nEMAIL:\n{email}"
    ),
    "masking": (
        "Rewrite the following email in a calm, professional business tone. Replace explicit "
        "warning signs (threats, 'urgent', 'verify your account', excessive punctuation) with "
        "softer, routine-sounding corporate phrasing while keeping the same request. "
        "Return only the email body.\n\nEMAIL:\n{email}"
    ),
    "personalization": (
        "Personalize the following email for the recipient {name} who works as {role} at "
        "{company}. Reference plausible internal details (team, project, manager) so that it "
        "reads like normal internal communication. Keep the same request. Return only the "
        "email body.\n\nEMAIL:\n{email}"
    ),
    # Gen-Gen needs "corresponding legitimate emails" generated by the LLM as well.
    "legit": (
        "Write a realistic, benign business email about: {topic}. It must not contain any "
        "request for credentials, payments or urgent action. Return only the email body."
    ),
}

NAMES = ["Alex Morgan", "Jamie Chen", "Taylor Brooks", "Jordan Lee", "Casey Nguyen",
         "Riley Patel", "Morgan Silva", "Sam Fischer", "Avery Kim", "Quinn Rossi"]
ROLES = ["financial analyst", "HR coordinator", "software engineer", "account manager",
         "procurement officer", "operations lead", "sales director", "IT administrator"]
COMPANIES = ["Northwind Ltd", "Contoso Group", "Fabrikam Inc", "Globex Corp",
             "Initech", "Umbrella Logistics", "Stark Supplies", "Wayne Partners"]
PROJECTS = ["Q3 budget review", "vendor onboarding", "the ERP migration",
            "annual compliance audit", "the Atlas rollout", "payroll consolidation"]


def local_llm_fn(model: str = "llama3", url: str = "http://localhost:11434/api/generate"):
    """Returns an llm_fn backed by a *local* Ollama server (nothing leaves the machine)."""

    def _call(prompt: str) -> str:
        body = json.dumps({"model": model, "prompt": prompt, "stream": False}).encode()
        req = urllib.request.Request(url, body, {"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=300) as r:
            return json.loads(r.read())["response"].strip()

    return _call


class LLMAugmenter:
    def __init__(self, llm_fn, seed: int = 0):
        self.llm_fn, self.rng = llm_fn, random.Random(seed)

    def __call__(self, email: str, behaviour: str) -> str:
        fields = dict(email=email, name=self.rng.choice(NAMES), role=self.rng.choice(ROLES),
                      company=self.rng.choice(COMPANIES))
        return self.llm_fn(PROMPTS[behaviour].format(**fields))

    def legit(self, topic: str) -> str:
        return self.llm_fn(PROMPTS["legit"].format(topic=topic))


# ---------------------------------------------------------------------------------------
# Offline rule-based stand-in
# ---------------------------------------------------------------------------------------
PARAPHRASE = {
    "click here": ["use the link below", "open the attached portal", "follow this secure link"],
    "verify": ["confirm", "validate", "re-check"],
    "account": ["profile", "workspace access", "user record"],
    "password": ["login credentials", "sign-in details", "access code"],
    "suspended": ["paused", "placed on hold", "temporarily restricted"],
    "immediately": ["at your earliest convenience", "today", "soon"],
    "winner": ["selected recipient", "eligible participant"],
    "invoice": ["billing statement", "payment summary"],
    "bank": ["financial institution", "payment provider"],
    "update": ["refresh", "review", "complete"],
}
MASKING = [
    (r"\burgent(ly)?\b", "time-sensitive"),
    (r"!+", "."),
    (r"\bact now\b", "please review when possible"),
    (r"\bwill be (closed|deleted|terminated)\b", "may need a quick review"),
    (r"\bfinal (warning|notice)\b", "friendly reminder"),
    (r"\bdear (customer|user|client)\b", "Hello"),
    (r"\bsecurity alert\b", "routine notice"),
    (r"\bfree\b", "complimentary"),
]


class RuleAugmenter:
    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)

    def paraphrase(self, email: str) -> str:
        out = email
        for k, alts in PARAPHRASE.items():
            out = re.sub(rf"\b{k}\b", lambda m: self.rng.choice(alts), out, flags=re.I)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", out) if s.strip()]
        if len(sents) > 3:  # light re-ordering of the middle sentences
            mid = sents[1:-1]
            self.rng.shuffle(mid)
            sents = [sents[0], *mid, sents[-1]]
        return " ".join(sents)

    def masking(self, email: str) -> str:
        out = email
        for pat, rep in MASKING:
            out = re.sub(pat, rep, out, flags=re.I)
        return out

    def personalization(self, email: str) -> str:
        name, role = self.rng.choice(NAMES), self.rng.choice(ROLES)
        comp, proj = self.rng.choice(COMPANIES), self.rng.choice(PROJECTS)
        sender = self.rng.choice([n for n in NAMES if n != name])
        first = name.split()[0]
        body = re.sub(r"^(dear [^,]*,|hello,?|hi,?)\s*", "", email, flags=re.I)
        return (f"Hi {first}, hope the {proj} is going well on your side. As the {role} "
                f"for {comp}, you are on the list for this. {body} "
                f"Thanks, {sender} - {comp}")

    def __call__(self, email: str, behaviour: str) -> str:
        return getattr(self, behaviour)(email)


BEHAVIOURS = ("paraphrase", "masking", "personalization")


def augment_corpus(phish_emails, augmenter, seed: int = 0, combos: bool = True):
    """Applies one behaviour (or a random chain of them) to every phishing email."""
    rng = random.Random(seed)
    out = []
    for e in phish_emails:
        chain = rng.sample(BEHAVIOURS, rng.randint(1, 3)) if combos else [rng.choice(BEHAVIOURS)]
        for b in chain:
            e = augmenter(e, b)
        out.append({"text": e, "behaviour": "+".join(chain)})
    return out

In [ ]:
%%writefile phishkd/data.py
"""Corpus handling (Section 3) and the five train/test scenarios (Section 5.2).

Unified corpus format (CSV): text,label,source
    label  : 1 = phishing, 0 = legitimate
    source : "orig" (real-world: Cambridge, Nazario, Chakraborty, Phishing Pot, Enron)
             "gen"  (LLM-generated / rewritten samples)

The real datasets are not redistributed here; `make_toy_corpus` builds a small synthetic
corpus with the same structure so the pipeline can be run end-to-end.
"""
import random

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

from .augment import COMPANIES, NAMES, PROJECTS, RuleAugmenter, augment_corpus

SCENARIOS = ("orig-orig", "gen-gen", "orig-gen", "gen-orig", "mixture")


def load_corpus(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    assert {"text", "label", "source"} <= set(df.columns), "need columns text,label,source"
    df = df.dropna(subset=["text"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    return df


# ---------------------------------------------------------------------------------------
# Toy corpus
# ---------------------------------------------------------------------------------------
_PHISH = [
    "Dear customer, your {svc} account has been suspended due to unusual activity. Click here "
    "to verify your password immediately or your account will be closed within {n} hours!",
    "URGENT: Security alert from {svc}. We detected a login from {country}. Verify your "
    "account now at {url} to avoid permanent suspension!!!",
    "Congratulations! You are the winner of a ${amt} gift card. Click here to claim your free "
    "prize. Act now, offer expires in {n} hours!",
    "Your {svc} invoice #{inv} is overdue. Update your bank details at {url} immediately to "
    "avoid a late penalty of ${amt}.",
    "Final warning: your mailbox storage is full. Click here to verify your password and "
    "upgrade your quota or all incoming email will be deleted.",
    "Dear user, the IT helpdesk requires you to update your login credentials. Your password "
    "expires today. Click here: {url}",
    "We could not deliver your package #{inv}. Pay the ${amt} customs fee at {url} or the "
    "parcel will be returned. Act now!",
    "Your {svc} payment was declined. Verify your card information immediately at {url} to "
    "keep your subscription active.",
]
_LEGIT = [
    "Hi team, the meeting about {proj} is moved to {day} at {hour}. Please bring the latest "
    "figures. Thanks, {name}",
    "Hello {name}, attached are the minutes from yesterday's discussion on {proj}. Let me "
    "know if I missed anything.",
    "Reminder: the office will be closed on {day} for maintenance. Remote work is encouraged. "
    "Facilities team",
    "Hi {name}, could you review the draft report for {proj} before {day}? Comments in the "
    "shared document are fine.",
    "Thanks for joining the {proj} kickoff. Next steps: finalize the scope and share the "
    "timeline by {day}.",
    "Lunch & learn on {day} at {hour}: an overview of {proj}. Snacks provided, no "
    "registration needed.",
    "Hi {name}, the contract with {comp} was signed today. Legal will circulate the final "
    "copy next week.",
    "Quarterly newsletter: highlights from {proj}, new hires in the {dept} team and upcoming "
    "events.",
]
_GEN_LEGIT_TOPICS = [
    "Hi {name}, following up on {proj}: I have updated the tracker and flagged two open "
    "items for the {dept} team. Happy to walk through them on {day}. Best, {sender}",
    "Hello {name}, as discussed with your manager, please find the agenda for the {proj} "
    "review on {day} at {hour}. No action needed beforehand. Regards, {sender} - {comp}",
    "Hi {name}, a quick note to confirm that the {dept} budget for {proj} was approved. "
    "Finance will share the breakdown in the usual channel. Thanks, {sender}",
    "Dear {name}, thank you for your feedback on {proj}. We incorporated your comments and "
    "will present the revised plan on {day}. Kind regards, {sender}",
]
_FILL = dict(
    svc=["PayPal", "Microsoft 365", "Netflix", "Amazon", "DHL", "Apple ID", "Chase", "Dropbox"],
    country=["Russia", "Brazil", "Nigeria", "China", "Romania"],
    url=["http://secure-login.verify-acct.com", "http://bit.ly/x9k2", "http://update-info.net/login",
         "http://account-check.co/auth"],
    day=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
    hour=["9am", "10:30", "2pm", "4pm"],
    dept=["finance", "HR", "engineering", "sales", "legal"],
    name=[n.split()[0] for n in NAMES], sender=NAMES, proj=PROJECTS, comp=COMPANIES,
)


def _fill(tpl, rng):
    vals = {k: rng.choice(v) for k, v in _FILL.items()}
    vals.update(n=rng.randint(2, 72), amt=rng.randint(50, 5000), inv=rng.randint(10000, 99999))
    return tpl.format(**vals)


def make_toy_corpus(n_orig: int = 600, n_gen: int = 600, phish_ratio: float = 0.5,
                    seed: int = 42, augmenter=None) -> pd.DataFrame:
    rng = random.Random(seed)
    augmenter = augmenter or RuleAugmenter(seed)
    rows = []
    n_op = int(n_orig * phish_ratio)
    rows += [(_fill(rng.choice(_PHISH), rng), 1, "orig", "") for _ in range(n_op)]
    rows += [(_fill(rng.choice(_LEGIT), rng), 0, "orig", "") for _ in range(n_orig - n_op)]
    # Gen: phishing seeds rewritten with paraphrase / masking / personalization
    n_gp = int(n_gen * phish_ratio)
    seeds = [_fill(rng.choice(_PHISH), rng) for _ in range(n_gp)]
    for r in augment_corpus(seeds, augmenter, seed=seed):
        rows.append((r["text"], 1, "gen", r["behaviour"]))
    rows += [(_fill(rng.choice(_GEN_LEGIT_TOPICS + _LEGIT), rng), 0, "gen", "")
             for _ in range(n_gen - n_gp)]
    df = pd.DataFrame(rows, columns=["text", "label", "source", "behaviour"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


# ---------------------------------------------------------------------------------------
# Scenarios + 5-fold CV with a 72 / 8 / 20 train / val / test partition
# ---------------------------------------------------------------------------------------
def scenario_folds(df: pd.DataFrame, scenario: str, n_folds: int = 5, val_frac: float = 0.1,
                   seed: int = 42):
    """Yields (fold, train_df, val_df, test_df).

    In-distribution (orig-orig, gen-gen, mixture): stratified K-fold over one pool; each fold
    holds out 20% as test and 10% of the rest (8% overall) as validation.
    Cross-distribution (orig-gen, gen-orig): K-fold run in parallel over both pools - train
    and validate on the train-domain folds, test on the matching fold of the other domain.
    """
    tr_src, te_src = {
        "orig-orig": ("orig", "orig"), "gen-gen": ("gen", "gen"),
        "orig-gen": ("orig", "gen"), "gen-orig": ("gen", "orig"),
        "mixture": (None, None),
    }[scenario]
    pick = lambda s: df if s is None else df[df.source == s]  # noqa: E731
    pool_tr = pick(tr_src).reset_index(drop=True)
    pool_te = pick(te_src).reset_index(drop=True)
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
    folds_tr = list(skf.split(pool_tr, pool_tr.label))
    folds_te = folds_tr if tr_src == te_src else list(skf.split(pool_te, pool_te.label))
    for k in range(n_folds):
        tr_idx, _ = folds_tr[k]
        _, te_idx = folds_te[k]
        tr = pool_tr.iloc[tr_idx]
        tr, va = train_test_split(tr, test_size=val_frac, stratify=tr.label, random_state=seed)
        yield k, tr.reset_index(drop=True), va.reset_index(drop=True), \
            pool_te.iloc[te_idx].reset_index(drop=True)


def set_seed(seed: int):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
%%writefile phishkd/analysis.py
"""Section 3.2 / 3.2.1 / 3.2.2 - corpus analysis tools.

* semantic proximity : average TF-IDF cosine similarity of every phishing email to all
                       legitimate emails (Fig. 2: original vs LLM-generated phishing)
* phishing-level     : question-driven scoring - each question is a soft signal, weighted
                       by its alignment with the ground truth, aggregated into one score
                       used to filter low-confidence synthetic samples
* deduplication      : remove near-duplicate emails by TF-IDF cosine similarity
"""
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score

from .text import clean_html


# ---------------------------------------------------------------------------------------
# Fig. 2 - similarity of phishing emails to legitimate emails
# ---------------------------------------------------------------------------------------
def similarity_to_legit(df: pd.DataFrame) -> pd.DataFrame:
    """Vocabulary is built over *all* emails; returns one row per phishing email with its
    average cosine similarity to the legitimate set (TF-IDF rows are L2-normalised, so the
    mean of dot products with the legit matrix = mean of row @ legit_centroid)."""
    vec = TfidfVectorizer(preprocessor=clean_html, stop_words="english")
    X = vec.fit_transform(df.text)
    legit = X[(df.label == 0).to_numpy()]
    centroid = np.asarray(legit.mean(axis=0)).ravel()
    phish_mask = (df.label == 1).to_numpy()
    sims = X[phish_mask] @ centroid
    out = df.loc[phish_mask, ["source"]].copy()
    out["avg_cosine_to_legit"] = sims
    return out


# ---------------------------------------------------------------------------------------
# 3.2.1 - question-driven phishing-level scoring
# ---------------------------------------------------------------------------------------
QUESTIONS = {
    "urgency": "Does this email convey a sense of urgency or panic? Answer 0-100.",
    "flattery": "How much flattery is present in this email? Answer 0-100.",
    "link_suspicion": "How suspicious is the embedded link? Answer 0-100.",
    "marketing": "How closely does this email resemble a marketing message? Answer 0-100.",
    "personal_details": "Does the email address the recipient by name or include overly "
                        "specific details? Answer 0-100.",
    "authority": "To what extent does the message imply consequences for inaction "
                 "(authority)? Answer 0-100.",
    "account_update": "Does the email request account updates or signature actions via a "
                      "link? Answer 0-100.",
    "click_pressure": "How strongly does the email pressure the recipient to click a link "
                      "urgently? Answer 0-100.",
    "scarcity": "Does the email suggest that an offer or opportunity is limited? Answer 0-100.",
}


class LLMQuestionScorer:
    """Asks every question to an LLM (`llm_fn(prompt) -> str`) and parses a 0-100 number."""

    def __init__(self, llm_fn):
        self.llm_fn = llm_fn

    def score(self, email: str) -> dict:
        res = {}
        for k, q in QUESTIONS.items():
            ans = self.llm_fn(f"{q}\nReply with a single number only.\n\nEMAIL:\n{email}")
            m = re.search(r"\d+(\.\d+)?", ans)
            res[k] = min(float(m.group()), 100.0) if m else 0.0
        return res


_LEX = {
    "urgency": r"urgent|immediately|now|today|expires?|within \d+ hours|asap|time-sensitive",
    "flattery": r"congratulations|valued|winner|selected|lucky|exclusive",
    "link_suspicion": r"https?://\S*(verify|login|secure|update|account|bit\.ly)\S*|click here",
    "marketing": r"offer|free|discount|prize|gift|deal|subscription",
    "personal_details": r"\bhi [A-Z][a-z]+|dear [A-Z][a-z]+|as the .* for",
    "authority": r"suspend|closed|deleted|terminated|penalty|returned|restricted|on hold",
    "account_update": r"verify|confirm|update|validate|credentials|password|bank details",
    "click_pressure": r"click|link|portal|follow",
    "scarcity": r"expires?|limited|last chance|only \d+|final",
}


class HeuristicQuestionScorer:
    """Offline stand-in for the LLM: each 'answer' = keyword hits mapped to 0-100."""

    def score(self, email: str) -> dict:
        return {k: min(100.0, 35.0 * len(re.findall(p, email, flags=re.I)))
                for k, p in _LEX.items()}


def phishing_level(df: pd.DataFrame, scorer, labels=None):
    """Returns (per-email answers + composite score, per-question weights).

    Weight of a question = its discriminative power w.r.t. the ground truth
    (AUC - 0.5, clipped at 0), so prompts aligned with phishing count more (cf. Fig. 3).
    """
    answers = pd.DataFrame([scorer.score(t) for t in df.text], index=df.index)
    labels = df.label if labels is None else labels
    weights = {}
    for k in answers:
        try:
            weights[k] = max(roc_auc_score(labels, answers[k]) - 0.5, 0.0)
        except ValueError:  # single class
            weights[k] = 1.0
    w = pd.Series(weights)
    w = w / w.sum() if w.sum() > 0 else pd.Series(1 / len(w), index=w.index)
    answers["phishing_level"] = answers[w.index] @ w
    return answers, w


def filter_low_confidence(df: pd.DataFrame, levels: pd.Series, thr: float = 20.0):
    """Drops synthetic *phishing* samples whose phishing-level score is below `thr`."""
    keep = ~((df.source == "gen") & (df.label == 1) & (levels < thr))
    return df[keep].reset_index(drop=True)


# ---------------------------------------------------------------------------------------
# 3.2.2 - near-duplicate removal
# ---------------------------------------------------------------------------------------
def deduplicate(df: pd.DataFrame, thr: float = 0.9, chunk: int = 2000) -> pd.DataFrame:
    """Greedy: keep an email unless it is >= thr cosine-similar to an earlier kept email."""
    X = TfidfVectorizer(preprocessor=clean_html).fit_transform(df.text)
    n = X.shape[0]
    drop = np.zeros(n, dtype=bool)
    for s in range(0, n, chunk):
        S = (X[s:s + chunk] @ X.T).tocsr()
        for i in range(S.shape[0]):
            gi = s + i
            if drop[gi]:
                continue
            row = S.getrow(i)
            dup = row.indices[(row.data >= thr) & (row.indices > gi)]
            drop[dup] = True
    return df[~drop].reset_index(drop=True)

In [ ]:
%%writefile phishkd/models.py
"""Section 4 - models.

Baselines (Word2Vec input, 1 sigmoid output, BCE):
    LSTMClassifier         - Sec 4.2, Eqs (1)-(4)
    BiLSTMClassifier       - Sec 4.3, Eqs (5)-(7)
    BiLSTMSingleHead       - Sec 4.4/4.5, Eqs (8)-(11)
    BiLSTMMultiHead        - Sec 4.6, Eqs (12)-(15), Table 2 (H=4, D=64 -> 256)
Teacher:
    build_teacher          - Sec 4.7, MobileBERT + 2-way classification head
Student (distilled):
    KDStudent              - Sec 4.8.1: WordPiece ids -> embedding initialised from the
                             teacher -> BiLSTM -> multi-head attention -> residual + LayerNorm
                             -> global average pooling -> dropout -> 2 logits
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def masked_mean(h, mask):
    m = mask.unsqueeze(-1).type_as(h)
    return (h * m).sum(1) / m.sum(1).clamp(min=1.0)


class MultiHeadAttention(nn.Module):
    """head_i = softmax(X W_i^Q (X W_i^K)^T / sqrt(d_k)) X W_i^V ; Output = Concat(heads) W^O + b^O.
    n_heads=1 gives the single-head attention of Eqs (8)-(9)."""

    def __init__(self, d_in, n_heads=4, d_head=64, d_out=None):
        super().__init__()
        self.h, self.d = n_heads, d_head
        self.q = nn.Linear(d_in, n_heads * d_head)
        self.k = nn.Linear(d_in, n_heads * d_head)
        self.v = nn.Linear(d_in, n_heads * d_head)
        self.o = nn.Linear(n_heads * d_head, d_out or d_in)

    def forward(self, x, mask=None, return_attn=False):
        B, T, _ = x.shape
        split = lambda t: t.view(B, T, self.h, self.d).transpose(1, 2)  # noqa: E731
        q, k, v = split(self.q(x)), split(self.k(x)), split(self.v(x))
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.d)          # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask[:, None, None, :], float("-inf"))
        attn = scores.softmax(-1)
        ctx = (attn @ v).transpose(1, 2).reshape(B, T, self.h * self.d)  # concat heads
        out = self.o(ctx)
        return (out, attn) if return_attn else out


class _W2VBase(nn.Module):
    """Frozen Word2Vec input layer: (B, T) ids -> (B, T, F=100)."""

    def __init__(self, embedding_matrix, freeze=True):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.as_tensor(embedding_matrix),
                                                freeze=freeze, padding_idx=0)


class LSTMClassifier(_W2VBase):
    def __init__(self, embedding_matrix, hidden=32, dropout=0.5, bidirectional=False):
        super().__init__(embedding_matrix)
        self.rnn = nn.LSTM(self.emb.embedding_dim, hidden, batch_first=True,
                           bidirectional=bidirectional)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), 1)

    def forward(self, ids):
        lengths = (ids != 0).sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(ids), lengths, batch_first=True,
                                                   enforce_sorted=False)
        _, (h_n, _) = self.rnn(packed)
        h = torch.cat([h_n[-2], h_n[-1]], -1) if self.rnn.bidirectional else h_n[-1]
        return self.fc(self.drop(h)).squeeze(-1)  # logit; sigmoid in the loss


class BiLSTMClassifier(LSTMClassifier):
    def __init__(self, embedding_matrix, hidden=32, dropout=0.5):
        super().__init__(embedding_matrix, hidden, dropout, bidirectional=True)


class BiLSTMAttention(_W2VBase):
    """BiLSTM (Table 2: 128-d output) -> attention -> pooled sentence vector -> dropout -> dense."""

    def __init__(self, embedding_matrix, hidden=64, n_heads=4, d_head=64, dropout=0.5):
        super().__init__(embedding_matrix)
        self.rnn = nn.LSTM(self.emb.embedding_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = MultiHeadAttention(2 * hidden, n_heads, d_head, d_out=2 * hidden)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, 1)

    def forward(self, ids, return_attn=False):
        mask = ids != 0
        mask[:, 0] |= ~mask.any(1)  # avoid all-masked rows for empty emails
        h, _ = self.rnn(self.emb(ids))
        a, w = self.attn(h, mask, return_attn=True)
        logit = self.fc(self.drop(masked_mean(a, mask))).squeeze(-1)
        return (logit, w) if return_attn else logit


class BiLSTMSingleHead(BiLSTMAttention):
    def __init__(self, embedding_matrix, hidden=64, dropout=0.5):
        super().__init__(embedding_matrix, hidden, n_heads=1, d_head=2 * hidden, dropout=dropout)


class BiLSTMMultiHead(BiLSTMAttention):
    def __init__(self, embedding_matrix, hidden=64, dropout=0.5):
        super().__init__(embedding_matrix, hidden, n_heads=4, d_head=64, dropout=dropout)


BASELINES = {
    "lstm": LSTMClassifier,
    "bilstm": BiLSTMClassifier,
    "bilstm_sh": BiLSTMSingleHead,
    "bilstm_mh": BiLSTMMultiHead,
}


# ---------------------------------------------------------------------------------------
# Teacher & student
# ---------------------------------------------------------------------------------------
TEACHER_NAME = "google/mobilebert-uncased"


def build_teacher(name: str = TEACHER_NAME):
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
    return tok, model


class KDStudent(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, n_heads=4, d_head=64,
                 dropout=0.5, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.attn = MultiHeadAttention(2 * hidden, n_heads, d_head, d_out=2 * hidden)
        self.norm = nn.LayerNorm(2 * hidden)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, 2)  # two-logit head for softmax-based distillation

    @classmethod
    def from_teacher(cls, teacher, **kw):
        """Eq (24): E_student = E_teacher (MobileBERT word-piece embedding, V x 128)."""
        E = teacher.get_input_embeddings().weight.detach().clone()
        pad = teacher.config.pad_token_id or 0
        student = cls(E.shape[0], E.shape[1], pad_id=pad, **kw)
        student.emb.weight.data.copy_(E)
        return student

    def forward(self, input_ids, attention_mask):
        mask = attention_mask.bool()
        h, _ = self.rnn(self.emb(input_ids))
        h = self.norm(h + self.attn(h, mask))  # residual normalisation
        return self.fc(self.drop(masked_mean(h, mask)))


def distillation_loss(z_s, z_t, y, alpha=0.5, tau=2.0):
    """Eqs (20)-(23): L = a * CE(y, z_S) + (1 - a) * tau^2 * KL(softmax(z_T/tau) || softmax(z_S/tau))."""
    hard = F.cross_entropy(z_s, y)
    soft = F.kl_div(F.log_softmax(z_s / tau, -1), F.softmax(z_t / tau, -1),
                    reduction="batchmean")
    return alpha * hard + (1 - alpha) * tau ** 2 * soft

In [ ]:
%%writefile phishkd/train.py
"""Training / evaluation loops.

* baselines  : Adam(lr=1e-3), BCE, batch 32, 5 epochs (Table 2)
* teacher    : MobileBERT fine-tuning, AdamW + linear warmup schedule, CE (Sec 4.7.3)
* KD student : Algorithm 1 - frozen teacher, Adam(lr=1e-4), alpha=0.5, tau=2, 3 epochs, batch 32
"""
import copy
import time
from dataclasses import dataclass

import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from .models import BASELINES, KDStudent, build_teacher, count_params, distillation_loss
from .text import Word2VecVocab, clean_html


@dataclass
class Config:
    batch_size: int = 32
    # baselines
    w2v_dim: int = 100
    w2v_max_len: int = 200
    base_lr: float = 1e-3
    base_epochs: int = 5
    # teacher
    teacher_name: str = "google/mobilebert-uncased"
    bert_max_len: int = 512          # paper: m <= 512; lower it on CPU
    teacher_lr: float = 5e-5
    teacher_epochs: int = 2
    teacher_min_steps: int = 120   # small corpora: add epochs until this many updates
    warmup_frac: float = 0.1
    max_grad_norm: float = 1.0
    # distillation
    kd_lr: float = 1e-4
    kd_epochs: int = 3
    alpha: float = 0.5
    tau: float = 2.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    verbose: bool = True


def metrics(y_true, y_pred) -> dict:
    return {
        "acc": 100 * accuracy_score(y_true, y_pred),
        "precision": 100 * precision_score(y_true, y_pred, zero_division=0),
        "recall": 100 * recall_score(y_true, y_pred, zero_division=0),
        "f1_weighted": 100 * f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def _loader(*tensors, batch_size, shuffle):
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)


def _trim(ids, mask):
    """Drop all-padding columns of a batch (dynamic padding)."""
    L = int(mask.sum(1).max().clamp(min=1))
    return ids[:, :L], mask[:, :L]


# ---------------------------------------------------------------------------------------
# Baselines: LSTM / BiLSTM / BiLSTM+SH / BiLSTM+MH
# ---------------------------------------------------------------------------------------
def run_baseline(name, tr, va, te, cfg: Config) -> dict:
    dev = cfg.device
    vocab = Word2VecVocab(cfg.w2v_dim, cfg.w2v_max_len).fit(tr.text.tolist())
    enc = lambda d: torch.as_tensor(vocab.encode(d.text.tolist()))  # noqa: E731
    lab = lambda d: torch.as_tensor(d.label.values, dtype=torch.float32)  # noqa: E731
    model = BASELINES[name](vocab.embedding_matrix).to(dev)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=cfg.base_lr)

    @torch.no_grad()
    def predict(ids):
        model.eval()
        out = [torch.sigmoid(model(b.to(dev))).cpu()
               for (b,) in _loader(ids, batch_size=256, shuffle=False)]
        return (torch.cat(out) >= 0.5).long().numpy()

    Xtr, ytr, Xva = enc(tr), lab(tr), enc(va)
    best, best_f1 = None, -1.0
    t0 = time.perf_counter()
    for ep in range(cfg.base_epochs):
        model.train()
        for xb, yb in _loader(Xtr, ytr, batch_size=cfg.batch_size, shuffle=True):
            loss = F.binary_cross_entropy_with_logits(model(xb.to(dev)), yb.to(dev))
            opt.zero_grad()
            loss.backward()
            opt.step()
        f1 = f1_score(va.label, predict(Xva), average="weighted")
        if f1 > best_f1:
            best_f1, best = f1, copy.deepcopy(model.state_dict())
    train_time = time.perf_counter() - t0
    model.load_state_dict(best)

    t0 = time.perf_counter()
    pred = predict(enc(te))
    test_time = time.perf_counter() - t0
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"params": trainable, "train_s": train_time, "test_s": test_time,
            **metrics(te.label, pred)}


# ---------------------------------------------------------------------------------------
# Teacher: fine-tune MobileBERT
# ---------------------------------------------------------------------------------------
def tokenize(tok, df, max_len):
    enc = tok([clean_html(t) for t in df.text], truncation=True, max_length=max_len,
              padding="max_length", return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


@torch.no_grad()
def predict_logits(model, ids, mask, cfg: Config, is_hf: bool):
    model.eval()
    out = []
    for i, m in _loader(ids, mask, batch_size=64, shuffle=False):
        i, m = _trim(i, m)
        z = model(input_ids=i.to(cfg.device), attention_mask=m.to(cfg.device))
        out.append((z.logits if is_hf else z).float().cpu())
    return torch.cat(out)


def train_teacher(tr, va, cfg: Config):
    from transformers import get_linear_schedule_with_warmup
    tok, model = build_teacher(cfg.teacher_name)
    model.to(cfg.device)
    ids, mask = tokenize(tok, tr, cfg.bert_max_len)
    y = torch.as_tensor(tr.label.values)
    vids, vmask = tokenize(tok, va, cfg.bert_max_len)
    dl = _loader(ids, mask, y, batch_size=cfg.batch_size, shuffle=True)
    epochs = max(cfg.teacher_epochs, -(-cfg.teacher_min_steps // len(dl)))
    steps = len(dl) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.teacher_lr)
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_frac * steps), steps)
    best, best_f1 = None, -1.0
    t0 = time.perf_counter()
    for ep in range(epochs):
        model.train()
        for i, m, yb in tqdm(dl, desc=f"teacher ep{ep + 1}", disable=not cfg.verbose):
            i, m = _trim(i, m)
            out = model(input_ids=i.to(cfg.device), attention_mask=m.to(cfg.device),
                        labels=yb.to(cfg.device))  # Eq (19): cross-entropy
            opt.zero_grad()
            out.loss.backward()
            # MobileBERT pools the raw [CLS] state (huge magnitude) -> initial loss ~1e7;
            # clipping (HF Trainer default max_grad_norm=1.0) keeps fine-tuning stable
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            opt.step()
            sched.step()
        vpred = predict_logits(model, vids, vmask, cfg, True).argmax(-1).numpy()
        f1 = f1_score(va.label, vpred, average="weighted")
        if f1 > best_f1:
            best_f1, best = f1, copy.deepcopy(model.state_dict())
    train_time = time.perf_counter() - t0
    model.load_state_dict(best)
    return tok, model, train_time


def evaluate_hf_or_student(model, tok, te, cfg: Config, is_hf: bool) -> dict:
    ids, mask = tokenize(tok, te, cfg.bert_max_len)
    t0 = time.perf_counter()
    pred = predict_logits(model, ids, mask, cfg, is_hf).argmax(-1).numpy()
    test_time = time.perf_counter() - t0
    return {"params": count_params(model), "test_s": test_time, **metrics(te.label, pred)}


# ---------------------------------------------------------------------------------------
# Algorithm 1 - knowledge distillation into BiLSTM + multi-head attention
# ---------------------------------------------------------------------------------------
def distill(teacher, tok, tr, va, cfg: Config, cache_teacher_logits: bool = True):
    dev = cfg.device
    for p in teacher.parameters():                       # 1: freeze teacher
        p.requires_grad_(False)
    teacher.eval()
    student = KDStudent.from_teacher(teacher).to(dev)    # Eq (24)
    opt = torch.optim.Adam(student.parameters(), lr=cfg.kd_lr)  # 2

    ids, mask = tokenize(tok, tr, cfg.bert_max_len)
    y = torch.as_tensor(tr.label.values)
    vids, vmask = tokenize(tok, va, cfg.bert_max_len)
    # The teacher is frozen and in eval mode, so its logits are a deterministic function of
    # the input: computing them once is equivalent to the per-batch forward of Algorithm 1
    # (line 5) and saves (E-1) teacher passes over the training set.
    t0 = time.perf_counter()
    z_t_all = predict_logits(teacher, ids, mask, cfg, True) if cache_teacher_logits else None
    idx = torch.arange(len(y))
    best, best_f1 = None, -1.0
    for ep in range(cfg.kd_epochs):                      # 3
        student.train()
        for (bi,) in tqdm(_loader(idx, batch_size=cfg.batch_size, shuffle=True),  # 4
                        desc=f"distill ep{ep + 1}", disable=not cfg.verbose):
            i, m = _trim(ids[bi], mask[bi])
            i, m, yb = i.to(dev), m.to(dev), y[bi].to(dev)
            if cache_teacher_logits:
                z_t = z_t_all[bi].to(dev)
            else:
                with torch.no_grad():
                    z_t = teacher(input_ids=i, attention_mask=m).logits      # 5
            z_s = student(i, m)                                              # 7
            loss = distillation_loss(z_s, z_t, yb, cfg.alpha, cfg.tau)       # 6, 8-11
            loss.backward()                                                  # 12
            opt.step()                                                       # 13
            opt.zero_grad()                                                  # 14
        vpred = predict_logits(student, vids, vmask, cfg, False).argmax(-1).numpy()
        f1 = f1_score(va.label, vpred, average="weighted")
        if f1 > best_f1:
            best_f1, best = f1, copy.deepcopy(student.state_dict())
    train_time = time.perf_counter() - t0
    student.load_state_dict(best)
    return student, train_time                           # 15


def summarize(rows):
    df = pd.DataFrame(rows)
    num = ["params", "train_s", "test_s", "acc", "precision", "recall", "f1_weighted"]
    return df.groupby(["model", "scenario"], sort=False)[num].mean().round(2).reset_index()

## 2. Tham số

- `INPUT_CSV`: để `None` thì dùng corpus toy; nếu có dữ liệu thật, upload file CSV có các cột `text,label,source` (`source` ∈ `orig`/`gen`) rồi điền đường dẫn vào đây.
- `MAX_FOLDS`: đặt `1` để chạy nhanh, `None` để chạy đủ 5 fold như bài báo.
- `BERT_MAX_LEN`: bài báo dùng 512; nếu chạy CPU hoặc muốn nhanh thì để 128.

In [ ]:
INPUT_CSV = None            # vd: '/content/my_corpus.csv'
N_ORIG, N_GEN = 1500, 1500  # kích thước corpus toy
SCENARIOS = ["orig-orig", "gen-gen", "orig-gen", "gen-orig", "mixture"]
BASELINES = ["lstm", "bilstm", "bilstm_sh", "bilstm_mh"]
RUN_TEACHER_KD = True
N_FOLDS, MAX_FOLDS = 5, 1
BERT_MAX_LEN = 128
TEACHER_EPOCHS, KD_EPOCHS, BASE_EPOCHS = 2, 3, 5
SEED = 42

## 3. Dữ liệu (Sec. 3 / Fig. 1)

In [ ]:
import pandas as pd
from phishkd.data import load_corpus, make_toy_corpus
from phishkd.augment import RuleAugmenter

if INPUT_CSV:
    df = load_corpus(INPUT_CSV)
else:
    df = make_toy_corpus(N_ORIG, N_GEN, augmenter=RuleAugmenter(SEED), seed=SEED)
print(len(df)); display(df.groupby(["source", "label"]).size().rename("n"))
df.sample(5, random_state=0)

### Fig. 2 — độ tương đồng cosine TF-IDF của email phishing so với email hợp lệ

In [ ]:
from phishkd.analysis import similarity_to_legit
sim = similarity_to_legit(df)
sim.groupby("source").avg_cosine_to_legit.describe().round(4)

### 3.2.1 — chấm điểm mức phishing bằng bộ câu hỏi
Trên Colab không có LLM nên dùng `HeuristicQuestionScorer`. Nếu có LLM thì thay bằng `LLMQuestionScorer(llm_fn)`, trong đó `llm_fn(prompt) -> str`.

In [ ]:
from phishkd.analysis import HeuristicQuestionScorer, phishing_level, filter_low_confidence
answers, weights = phishing_level(df, HeuristicQuestionScorer())
display(weights.sort_values(ascending=False).round(3).rename("weight"))
n = len(df)
df = filter_low_confidence(df, answers.phishing_level, thr=20.0)
print("removed low-confidence synthetic phishing:", n - len(df))

### 3.2.2 — khử trùng lặp

In [ ]:
from phishkd.analysis import deduplicate
n = len(df)
df = deduplicate(df, thr=0.95)
print(f"removed {n - len(df)} near-duplicates -> {len(df)}")
display(df.groupby(["source", "label"]).size().rename("n"))
df.to_csv("data/corpus.csv", index=False)

In [ ]:
from phishkd.data import scenario_folds, set_seed
from phishkd.train import (Config, distill, evaluate_hf_or_student, run_baseline,
                           summarize, train_teacher)

cfg = Config(bert_max_len=BERT_MAX_LEN, teacher_epochs=TEACHER_EPOCHS,
             kd_epochs=KD_EPOCHS, base_epochs=BASE_EPOCHS)
print("device:", cfg.device)
rows = []

def run(models, teacher_kd):
    for scen in SCENARIOS:
        for k, tr, va, te in scenario_folds(df, scen, N_FOLDS, seed=SEED):
            if MAX_FOLDS is not None and k >= MAX_FOLDS:
                break
            print(f"=== {scen} | fold {k + 1}/{N_FOLDS} | train {len(tr)} val {len(va)} test {len(te)}")
            for name in models:
                set_seed(SEED + k)
                r = run_baseline(name, tr, va, te, cfg)
                rows.append({"model": name, "scenario": scen, "fold": k, **r})
                print(f"  {name:12s} F1={r['f1_weighted']:.2f}")
            if teacher_kd:
                set_seed(SEED + k)
                tok, teacher, t_train = train_teacher(tr, va, cfg)
                r = {"train_s": t_train, **evaluate_hf_or_student(teacher, tok, te, cfg, True)}
                rows.append({"model": "mobilebert", "scenario": scen, "fold": k, **r})
                print(f"  mobilebert   F1={r['f1_weighted']:.2f}")
                student, s_train = distill(teacher, tok, tr, va, cfg)
                r = {"train_s": s_train, **evaluate_hf_or_student(student, tok, te, cfg, False)}
                rows.append({"model": "kd_bilstm_mh", "scenario": scen, "fold": k, **r})
                print(f"  kd_bilstm_mh F1={r['f1_weighted']:.2f}")
                del teacher, student
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

### 4.1 Baselines (Tables 4–5)

In [ ]:
run(BASELINES, teacher_kd=False)
summarize(rows)

### 4.2 Teacher MobileBERT + KD-BiLSTM (Table 6, Algorithm 1)

In [ ]:
cfg.verbose = False
if RUN_TEACHER_KD:
    run([], teacher_kd=True)
summary = summarize(rows)
summary

## 5. Lưu kết quả

In [ ]:
pd.DataFrame(rows).to_csv("results/results.csv", index=False)
summary.to_csv("results/results_summary.csv", index=False)
# Bảng giống Table 7: weighted F1 theo mô hình × kịch bản
summary.pivot(index="model", columns="scenario", values="f1_weighted")